In [38]:
# imports
from huggingface_hub import snapshot_download
from sklearn.model_selection import train_test_split
import pandas as pd
import os

In [4]:
repo_id = "hu-lab/TFBS_Prediction"
repo_type = "dataset"
local_dir = "../data/raw"

In [ ]:
# download dataset
snapshot_download(repo_id = repo_id, repo_type = repo_type, local_dir = local_dir)

In [33]:
%%bash

# remove spaces and merge raw data
dataset_types=("test" "train" "val")
for type in "${dataset_types[@]}"; do
    echo "sequence,labels" > ../data/merged/${type}.csv
    grep -h -v "sequence" ../data/raw/*/${type}.csv | tr -d '[:blank:]' >> ../data/merged/${type}.csv
done

In [34]:
# check columns after merging
df = pd.read_csv("../data/merged/val.csv")
print(df.columns.to_list())

['sequence', 'labels']


In [35]:
dataset_types = ["test", "train", "val"]
merged_dir = "../data/merged"

In [36]:
# split X and y then remove pre-split files
for type in dataset_types:
    df = pd.read_csv(f"{merged_dir}/{type}.csv")
    df_X = df["sequence"]
    df_y = df["labels"]

    df_X.to_csv(f"{merged_dir}/X_{type}.csv", index = False)
    df_y.to_csv(f"{merged_dir}/y_{type}.csv", index = False)

    os.remove(f"{merged_dir}/{type}.csv")

In [37]:
random_state = 23

In [39]:
# get 10% of test, train, val
for type in dataset_types:
    df_X = df = pd.read_csv(f"{merged_dir}/X_{type}.csv")
    df_y = df = pd.read_csv(f"{merged_dir}/y_{type}.csv")

    small_X, _, small_y, _ = train_test_split(
        df_X,
        df_y,
        train_size = 0.1,
        stratify = df_y,
        random_state = random_state
    )

    small_X.to_csv(f"{merged_dir}/small_X_{type}.csv", index = False)
    small_y.to_csv(f"{merged_dir}/small_y_{type}.csv", index = False)